In [1]:
import io
import urllib.request
import re
import pandas as pd

# Core translation and molecular mass reference tables
Amino_Acid_Codon_Table = {
    'UUU':'F','UUC': 'F', 'UUA': 'L', 'UUG': 'L', 'UCU': 'S', 'UCC': 'S', 'UCA': 'S', 'UCG': 'S',
    'UAU': 'Y', 'UAC': 'Y', 'UAA': 'Stop', 'UAG': 'Stop', 'UGU': 'C', 'UGC': 'C', 'UGA': 'Stop', 'UGG': 'W',
    'CUU': 'L', 'CUC': 'L', 'CUA': 'L', 'CUG': 'L', 'CCU': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAU': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q', 'CGU': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AUU': 'I', 'AUC': 'I', 'AUA': 'I', 'AUG': 'M', 'ACU': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAU': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K', 'AGU': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GUU': 'V', 'GUC': 'V', 'GUA': 'V', 'GUG': 'V', 'GCU': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAU': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E', 'GGU': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

Mono_Isotopic_Mass = {
    'A': 71.03711, 'C': 103.00919, 'D': 115.02694, 'E': 129.04259, 'F': 147.06841,
    'G': 57.02146, 'H': 137.05891, 'I': 113.08406, 'K': 128.09496, 'L': 113.08406,
    'M': 131.04049, 'N': 114.04293, 'P': 97.05276, 'Q': 128.05858, 'R': 156.10111,
    'S': 87.03203, 'T': 101.04768, 'V': 99.06841, 'W': 186.07931, 'Y': 163.06333
}

# Bidirectional mapping engine maps for 3-letter to 1-letter translations
Triple_Code_to_Single = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C', 'Gln': 'Q', 'Glu': 'E',
    'Gly': 'G', 'His': 'H', 'Ile': 'I', 'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F',
    'Pro': 'P', 'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'
}

# Dictionary comprehension dynamically reverses the mapping structure 
Single_Code_to_Triple = {value: key for key, value in Triple_Code_to_Single.items()}


In [2]:
def UNIPROT_fasta_parser(file_stream, expected_id):
    """Parses an in-memory FASTA stream and associates records with the target accession ID."""
    sequences = {expected_id: ''}
    for line in file_stream:
        line = line.strip()
        if not line or line.startswith('>'):
            continue
        sequences[expected_id] += line
    return sequences


In [3]:
def ntcount_optimized(text):
    """Calculates absolute nucleotide distributions directly without nested loop parsing."""
    DNA = text.upper()
    return pd.DataFrame([{
        'A': DNA.count('A'), 'C': DNA.count('C'),
        'G': DNA.count('G'), 'T': DNA.count('T')
    }])

def complement_fast(pattern):
    """Generates RNA transcription matches via array joining to save systems memory."""
    comp_dict = {'A': 'U', 'T': 'A', 'C': 'G', 'G': 'C'}
    return "".join(comp_dict[ntd] for ntd in pattern)

def Reverse(pattern):
    """Reverses the character order of sequence slices using slicing logic."""
    return pattern[::-1]

def Protein_Mass_Monoisotopic(protein_sequence):
    """Computes exact peptide mass vectors using table lookup accumulations."""
    return sum(Mono_Isotopic_Mass[amino_acid] for amino_acid in protein_sequence)

def convert_amino_acid_notation(raw_sequence, target_format='single'):
    """
    Translates sequences between 3-letter abbreviations and 1-letter characters.
    Handles text inputs with dashes, without dashes, or passed as clean arrays.
    """
    if target_format == 'single':
        if isinstance(raw_sequence, str):
            clean_str = raw_sequence.strip()
            # Tunnel A: String has dashes (e.g., "ala-phe")
            if '-' in clean_str:
                elements = clean_str.split('-')
            # Tunnel B: Continuous string without dashes (e.g., "alaphe")
            else:
                elements = [clean_str[i:i+3] for i in range(0, len(clean_str), 3)]
        else:
            elements = raw_sequence
            
        # .title() normalizes text ('ala' or 'ALA' -> 'Ala') for dictionary lookup
        return "".join(Triple_Code_to_Single.get(amino.strip().title(), '?') for amino in elements)
        
    elif target_format == 'triple':
        if isinstance(raw_sequence, str):
            # Strips spacing/dashes out entirely to read individual single characters
            elements = [char for char in raw_sequence if char not in ('-', ' ')]
        else:
            elements = raw_sequence
            
        # .upper() normalizes text ('a' -> 'A') for reversed dictionary lookup
        return "-".join(Single_Code_to_Triple.get(char.upper(), '???') for char in elements)


In [4]:
def Hamming_Dist(s, t):
    """Measures mismatched character coordinates across equivalent sequence arrays."""
    return sum(1 for n in range(len(s)) if s[n] != t[n])

def rosalind_motif_converter(rosalind_string):
    """Translates bracket strings into standard compiled Python regex patterns."""
    regex_string = rosalind_string.replace('{', '[^').replace('}', ']')
    return re.compile(regex_string)

def rosalind_protein_scanner(id_registry, sequence_database, raw_motif):
    """Runs a 1-step sliding window scan across target records using looking assertions."""
    regex_engine = rosalind_motif_converter(raw_motif)
    for protein_id in id_registry:
        sequence = sequence_database.get(protein_id)
        if not sequence:
            continue
        
        match_coordinates = []
        for i in range(len(sequence)):
            if regex_engine.match(sequence[i:]):
                match_coordinates.append(i + 1)  # Convert to 1-based indexing for biology
        
        if match_coordinates:
            print(f"ID: {protein_id} | Motif Matches Found at Positions: {' '.join(str(coord) for coord in match_coordinates)}")


In [ ]:
# Live system integration verification test with local fallback logic
test_id = 'B5ZC00'
url = f"https://rest.uniprot.org/uniprotkb/{test_id}?fromat=fasta"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

try:
    # Try fetching from the web first
    with urllib.request.urlopen(req) as response:
        raw_text = response.read().decode('utf-8')
    print(f"Network online. Successfully streamed {test_id} from UniProt API.")
    
except Exception as network_error:
    # If the Wi-Fi is broken/offline, use this local fallback data automatically
    print(f"Network offline ({network_error}). Switching to local fallback stream...")
    raw_text = f">tr|{test_id}|{test_id}_MOCK Mock protein sequence\nMETHIONINEALANINEPHENYLALANINE"

# The rest of your input-output schema executes exactly the same way
fasta_file = io.StringIO(raw_text)
parsed_data = UNIPROT_fasta_parser(fasta_file, test_id)
print(f"Sequence Length: {len(parsed_data[test_id])} amino acids.")

# Test your bidirectional conversion mechanics
print("Notation Check 1 (ala-phe):", convert_amino_acid_notation("ala-phe", 'single'))
print("Notation Check 2 (ALAPHE):", convert_amino_acid_notation("ALAPHE", 'single'))
print("Notation Check 3 (af):     ", convert_amino_acid_notation("af", 'triple'))
